# <font color="#418FDE" size="6.5" uppercase>**Imputation Targets**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Apply univariate, multivariate, and neighbor-based imputation to incomplete data. 
- Use missingness indicators and understand estimators with native missing-value support. 
- Transform classification and regression targets using scikit-learn target utilities. 


## **1. Imputation Methods**

### **1.1. Basic Value Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_01.jpg?v=1783908183" width="250">



>* Fill blanks with fixed substitute values
>* Simple, fast, and model-friendly approach

>* Match imputation to data type and distribution
>* Use unknown categories when missingness matters

>* Can distort variability and feature relationships
>* Use sparingly; fit only on training data



In [ ]:
#@title Python Code - Basic Value Imputation

# Basic imputation fills missing values simply.
# This example uses tiny mixed data.
# We compare mean, median, and mode.

import numpy as np
import pandas as pd

# Create a small dataset with missing entries.
data = pd.DataFrame({
    "age_years": [29, 41, np.nan, 35, 52, np.nan],
    "height_cm": [170, np.nan, 165, 180, 172, 168],
    "blood_type": ["A", "O", None, "A", "B", None]
})

# Count missing values before imputation.
missing_before = data.isna().sum()

# Copy data before applying replacement rules.
filled = data.copy()

# Fill numeric age using the median.
filled["age_years"] = filled["age_years"].fillna(
    filled["age_years"].median()
)

# Fill numeric height using the mean.
filled["height_cm"] = filled["height_cm"].fillna(
    filled["height_cm"].mean()
)

# Fill categorical blood type using unknown.
filled["blood_type"] = filled["blood_type"].fillna(
    "Unknown"
)

# Count missing values after imputation.
missing_after = filled.isna().sum()

# Show compact results for learners.
print("Original missing counts:", missing_before.to_dict())
print("Filled missing counts:", missing_after.to_dict())
print("Age median used:", round(data["age_years"].median(), 1))
print("Height mean used:", round(data["height_cm"].mean(), 1))
print("Blood type fill used: Unknown")
print("Imputed preview:")
print(filled.head(6).to_string(index=False))



### **1.2. Iterative Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_02.jpg?v=1783908181" width="250">



>* Predict missing values using other features
>* Model each incomplete variable in turn

>* Repeatedly updates predictions for each incomplete feature
>* Useful when variables have overlapping missing values

>* Preserves patterns but needs careful modeling
>* Imputed values are useful estimates, not facts



### **1.3. Nearest Neighbor Imputation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_01_03.jpg?v=1783908190" width="250">



>* Fill gaps using similar records
>* Estimate values from nearby cases

>* Preserves subgroup patterns with context-aware estimates
>* Requires well-chosen, properly scaled similarity features

>* Can be slow and data-sensitive
>* Works best with preprocessing and validation



## **2. Missingness Signals**

### **2.1. Missingness Indicators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_01.jpg?v=1783908174" width="250">



>* Flags values that were originally missing
>* Preserves missingness as a predictive signal

>* Missing patterns can reveal real-world causes
>* Indicators separate observed values from imputations

>* Use indicators carefully; they add complexity
>* Check signals for bias and generalization



In [ ]:
#@title Python Code - Missingness Indicators

# Missingness indicators preserve absence information.
# Tiny data keeps the example clear.
# Built-in tools avoid extra dependencies.

import math
import statistics

# Store heights with some missing values.
heights_cm = [170, None, 165, 180, None, 172]

# Store outcomes related to missingness.
risk_score = [1, 4, 1, 2, 5, 2]

# Validate matching list lengths before processing.
assert len(heights_cm) == len(risk_score)

# Compute the mean from observed heights.
observed = [value for value in heights_cm if value is not None]
mean_height = statistics.mean(observed)

# Impute missing heights and create indicators.
imputed = [mean_height if value is None else value for value in heights_cm]
missing_flag = [1 if value is None else 0 for value in heights_cm]

# Compare average risk by missingness group.
missing_risks = [risk_score[i] for i, flag in enumerate(missing_flag) if flag]
observed_risks = [risk_score[i] for i, flag in enumerate(missing_flag) if not flag]

# Print compact teaching output only.
print("Original heights:", heights_cm)
print("Imputed heights:", [round(value, 1) for value in imputed])
print("Missing indicators:", missing_flag)
print("Mean risk when observed:", round(statistics.mean(observed_risks), 2))
print("Mean risk when missing:", round(statistics.mean(missing_risks), 2))
print("Indicator keeps the missingness signal visible.")



### **2.2. Empty Feature Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_02.jpg?v=1783908172" width="250">



>* Features may be entirely unobserved during fitting
>* Imputation lacks values to learn replacements

>* Dropping empty features changes model input shape
>* Placeholders preserve stable production schemas

>* Missingness indicators can reveal data collection signals
>* Check for artifacts, bias, and leakage



In [ ]:
#@title Python Code - Empty Feature Handling

# Empty columns need deliberate handling.
# Missingness indicators preserve useful signals.
# Stable schemas help production predictions.

import numpy as np
import pandas as pd

# Create tiny training and future datasets.
train = pd.DataFrame({
    "age": [34, 52, np.nan, 41],
    "lab_score": [np.nan, np.nan, np.nan, np.nan],
    "height_cm": [170, np.nan, 165, 180],
})

future = pd.DataFrame({
    "age": [45, np.nan],
    "lab_score": [7.2, 8.1],
    "height_cm": [172, np.nan],
})

# Identify columns empty during fitting.
empty_cols = train.columns[train.isna().all()].tolist()
observed_cols = train.columns.difference(empty_cols).tolist()

# Compute simple means only where possible.
means = train[observed_cols].mean(numeric_only=True)

# Keep schema by filling empty columns with constants.
train_filled = train.copy()
future_filled = future.copy()

# Add indicators before replacing missing values.
for col in train.columns:
    train_filled[col + "_was_missing"] = train[col].isna().astype(int)
    future_filled[col + "_was_missing"] = future[col].isna().astype(int)

# Fill observed columns using learned training means.
train_filled[observed_cols] = train[observed_cols].fillna(means)
future_filled[observed_cols] = future[observed_cols].fillna(means)

# Fill empty training columns with stable placeholders.
train_filled[empty_cols] = train[empty_cols].fillna(0.0)
future_filled[empty_cols] = future[empty_cols].fillna(0.0)

# Validate that both transformed schemas match.
same_schema = list(train_filled.columns) == list(future_filled.columns)

# Print compact teaching summary only.
print("Empty during fit:", empty_cols)
print("Learned means:", means.round(1).to_dict())
print("Same transformed schema:", same_schema)
print("Train shape before/after:", train.shape, train_filled.shape)
print("Future lab values kept:", future_filled["lab_score"].tolist())



### **2.3. Native Missing Support**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_02_03.jpg?v=1783908170" width="250">



>* Some models handle missing values directly
>* Missingness can reveal useful outcome signals

>* Trees can route missing values during splits
>* Check missingness meaning and stability carefully

>* Simplifies preprocessing while preserving missingness signals
>* Requires validation, monitoring, and domain knowledge



## **3. Target Transformations**

### **3.1. Encoding Class Labels**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_01.jpg?v=1783908195" width="250">



>* Convert target class names into integers
>* Use encoding for labels, not features

>* Encoded labels are identifiers, not measurements
>* Convert predictions back to original class names

>* Reuse training label mappings across workflows
>* Encoding standardizes labels, not data issues



In [ ]:
#@title Python Code - Encoding Class Labels

# Encode class labels for classification targets.
# Keep numeric codes as identifiers only.
# Decode predictions back to original labels.

# Import small, standard teaching libraries.
import numpy as np

# Create a tiny target vector.
y_labels = np.array([
    "approved", "denied", "approved",
    "review", "denied", "approved"
])

# Validate the target is one-dimensional.
assert y_labels.ndim == 1

# Learn a stable alphabetical class mapping.
classes = np.unique(y_labels)

# Build dictionaries for encoding and decoding.
label_to_code = {label: index for index, label in enumerate(classes)}
code_to_label = {index: label for label, index in label_to_code.items()}

# Encode each original class label.
y_encoded = np.array([label_to_code[label] for label in y_labels])

# Pretend these are model output codes.
predicted_codes = np.array([0, 2, 1])

# Decode numeric predictions for reporting.
predicted_labels = np.array([code_to_label[code] for code in predicted_codes])

# Print the learned mapping compactly.
print("Class mapping:", label_to_code)

# Print original and encoded targets.
print("Original labels:", y_labels.tolist())
print("Encoded target:", y_encoded.tolist())

# Print decoded predictions for humans.
print("Decoded predictions:", predicted_labels.tolist())
print("Codes are identifiers, not ordered measurements.")



### **3.2. Multilabel Binarization**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_02.jpg?v=1783908192" width="250">



>* Examples can have multiple labels
>* Labels become binary target columns

>* Handles records with overlapping labels
>* Enables per-label learning and evaluation

>* Standardize labels across training and future data
>* Handle sparse labels and estimator suitability



In [ ]:
#@title Python Code - Multilabel Binarization

# Multilabel targets can contain several labels.
# Binarization creates one column per label.
# This example uses pure Python only.

# No extra installations are required.

# Store sample records with multiple tags.
records = [
    ["billing", "delivery"],
    ["product_quality"],
    ["delivery", "urgent"],

    ["billing", "urgent"],
    ["product_quality", "delivery"],
]

# Collect unique labels in stable order.
labels = sorted({tag for row in records for tag in row})

# Validate that records and labels exist.
assert len(records) > 0 and len(labels) > 0

# Convert each label collection into binary values.
binary_rows = []
for row in records:
    row_set = set(row)

    binary_row = [1 if label in row_set else 0 for label in labels]
    binary_rows.append(binary_row)

# Validate the rectangular target matrix shape.
assert all(len(row) == len(labels) for row in binary_rows)

# Print compact teaching output only.
print("Labels:", labels)
print("First original target:", records[0])
print("First binarized target:", binary_rows[0])
print("Matrix shape:", (len(binary_rows), len(labels)))
print("Each column is a yes-or-no target.")



### **3.3. Transformed Targets**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_B/image_03_03.jpg?v=1783908202" width="250">



>* Transform skewed targets for easier learning
>* Invert predictions back to original units

>* Log transforms balance skewed regression targets
>* Inverse transforms restore meaningful original units

>* Interpret transformed-target errors carefully
>* Use wrappers to invert predictions consistently



# <font color="#418FDE" size="6.5" uppercase>**Imputation Targets**</font>


In this lecture, you learned to:
- Apply univariate, multivariate, and neighbor-based imputation to incomplete data. 
- Use missingness indicators and understand estimators with native missing-value support. 
- Transform classification and regression targets using scikit-learn target utilities. 

In the next Module (Module 6), we will go over 'None'